In [ ]:
dia = '2024-04-27'

### Instalando o PyCaret

In [ ]:
!pip install pycaret

### importando as Bibliotecas e Funções

In [ ]:
import pandas as pd
import numpy as np

from pycaret.classification import *

import warnings
warnings.filterwarnings('ignore')

def drop_reset_index(df):
    df = df.dropna()
    df = df.reset_index(drop=True)
    df.index += 1
    return df

def plot_profit_acu(dataframe, title_text):
    dataframe['Profit_acu'] = dataframe.Profit.cumsum()
    n_apostas = dataframe.shape[0]
    profit = round(dataframe.Profit_acu.tail(1).item(), 2)
    ROI = round((dataframe.Profit_acu.tail(1) / n_apostas * 100).item(), 2)
    drawdown = dataframe['Profit_acu'] - dataframe['Profit_acu'].cummax()
    drawdown_maximo = round(drawdown.min(), 2)
    winrate_medio = round((dataframe['Profit'] > 0).mean() * 100, 2)
    desvio_padrao = round(dataframe['Profit'].std(), 2)
    dataframe.Profit_acu.plot(title=title_text, xlabel='Entradas', ylabel='Stakes')
    print("Metodo:",title_text)
    print("Profit:", profit, "stakes em", n_apostas, "jogos")
    print("ROI:", ROI, "%")
    print("Drawdown Maximo Acumulado:", drawdown_maximo)
    print("Winrate Medio:", winrate_medio, "%")
    print("Desvio Padrao:", desvio_padrao)
    print("")

def read_base():
    base = pd.read_csv("https://github.com/futpythontrader/YouTube/raw/main/Luke/Bases_de_Dados/Base_FutPythonPunter.csv")
    base["Date"] = pd.to_datetime(base["Date"])
    base = base[['League','Date','Home','Away','Goals_H','Goals_A','Odd_H','Odd_D','Odd_A','Odd_Over25','Odd_Under25','Odd_BTTS_Yes','Odd_BTTS_No','Goals_Minutes_Home','Goals_Minutes_Away']]
    base = drop_reset_index(base)
    return base

def read_jogos(dia):
    jogos_do_dia = pd.read_csv('https://github.com/futpythontrader/YouTube/blob/main/Jogos_do_Dia_FlashScore/'+str(dia)+'_Jogos_do_Dia_FlashScore.csv?raw=true')
    jogos_do_dia = jogos_do_dia[['League','Date','Time','Home','Away','Odd_H','Odd_D','Odd_A','Odd_Over25','Odd_Under25','Odd_BTTS_Yes','Odd_BTTS_No']]
    jogos_do_dia = drop_reset_index(jogos_do_dia)
    return jogos_do_dia

### Importando os Jogos do Dia e a Base de Dados

In [ ]:
Jogos_do_Dia = read_jogos(dia)
not_leagues = ['SOUTH AFRICA - PREMIER LEAGUE']
Jogos_do_Dia = Jogos_do_Dia[Jogos_do_Dia['League'].isin(not_leagues) == False]
Jogos_do_Dia = Jogos_do_Dia.sort_values(by='League')
leagues = Jogos_do_Dia['League'].unique()
leagues

In [ ]:
base_orig = read_base()
not_leagues = ['SOUTH AFRICA - PREMIER LEAGUE']
base_orig = base_orig[base_orig['League'].isin(not_leagues) == False]
base_orig["Date"] = pd.to_datetime(base_orig["Date"])
base_orig = base_orig.sort_values('Date')

flt = base_orig.Date < str(dia)
base = base_orig[flt]

### Modelo 01


In [ ]:
leagues = ['ENGLAND - PREMIER LEAGUE']
base = base[base['League'].isin(leagues) == True]
base = drop_reset_index(base)

In [ ]:
n_per = 5

base['p_H'] = 1 / base['Odd_H']
base['p_D'] = 1 / base['Odd_D']
base['p_A'] = 1 / base['Odd_A']

desvio_padrao = base[['Odd_H','Odd_D','Odd_A']].std(axis=1)
media = base[['Odd_H','Odd_D','Odd_A']].mean(axis=1)
CV_Odds = desvio_padrao / media
base['CV_Odds'] = CV_Odds

base['Ptos_H'] = np.where(base['Goals_H'] >  base['Goals_A'], 3,
               np.where(base['Goals_H'] == base['Goals_A'], 1, 0))

base['Ptos_A'] = np.where(base['Goals_H'] > base['Goals_A'], 0,
               np.where(base['Goals_H'] == base['Goals_A'], 1, 3))
base['Media_Ptos_H'] = base.groupby('Home')['Ptos_H'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['Media_Ptos_A'] = base.groupby('Away')['Ptos_A'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['Media_Ptos_H'] = base.groupby('Home')['Media_Ptos_H'].shift(1)
base['Media_Ptos_A'] = base.groupby('Away')['Media_Ptos_A'].shift(1)
base['DesvPad_Ptos_H'] = base.groupby('Home')['Ptos_H'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['DesvPad_Ptos_A'] = base.groupby('Away')['Ptos_A'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['DesvPad_Ptos_H'] = base.groupby('Home')['DesvPad_Ptos_H'].shift(1)
base['DesvPad_Ptos_A'] = base.groupby('Away')['DesvPad_Ptos_A'].shift(1)
base['CV_Ptos_H'] = base['DesvPad_Ptos_H'] / base['Media_Ptos_H']
base['CV_Ptos_A'] = base['DesvPad_Ptos_A'] / base['Media_Ptos_A']

base['Media_GM_H'] = base.groupby('Home')['Goals_H'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['Media_GM_A'] = base.groupby('Away')['Goals_A'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['Media_GM_H'] = base.groupby('Home')['Media_GM_H'].shift(1)
base['Media_GM_A'] = base.groupby('Away')['Media_GM_A'].shift(1)
base['DesvPad_GM_H'] = base.groupby('Home')['Goals_H'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['DesvPad_GM_A'] = base.groupby('Away')['Goals_A'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['DesvPad_GM_H'] = base.groupby('Home')['DesvPad_GM_H'].shift(1)
base['DesvPad_GM_A'] = base.groupby('Away')['DesvPad_GM_A'].shift(1)
base['CV_GM_H'] = base['DesvPad_GM_H'] / base['Media_GM_H']
base['CV_GM_A'] = base['DesvPad_GM_A'] / base['Media_GM_A']

base['Media_GS_H'] = base.groupby('Home')['Goals_A'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['Media_GS_A'] = base.groupby('Away')['Goals_H'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['Media_GS_H'] = base.groupby('Home')['Media_GS_H'].shift(1)
base['Media_GS_A'] = base.groupby('Away')['Media_GS_A'].shift(1)
base['DesvPad_GS_H'] = base.groupby('Home')['Goals_A'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['DesvPad_GS_A'] = base.groupby('Away')['Goals_H'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['DesvPad_GS_H'] = base.groupby('Home')['DesvPad_GS_H'].shift(1)
base['DesvPad_GS_A'] = base.groupby('Away')['DesvPad_GS_A'].shift(1)
base['CV_GS_H'] = base['DesvPad_GS_H'] / base['Media_GS_H']
base['CV_GS_A'] = base['DesvPad_GS_A'] / base['Media_GS_A']

base['SG_H'] = base['Goals_H'] - base['Goals_A']
base['SG_A'] = base['Goals_A'] - base['Goals_H']
base['Media_SG_H'] = base.groupby('Home')['SG_H'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['Media_SG_A'] = base.groupby('Away')['SG_A'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['Media_SG_H'] = base.groupby('Home')['Media_SG_H'].shift(1)
base['Media_SG_A'] = base.groupby('Away')['Media_SG_A'].shift(1)
base['DesvPad_SG_H'] = base.groupby('Home')['SG_H'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['DesvPad_SG_A'] = base.groupby('Away')['SG_A'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['DesvPad_SG_H'] = base.groupby('Home')['DesvPad_SG_H'].shift(1)
base['DesvPad_SG_A'] = base.groupby('Away')['DesvPad_SG_A'].shift(1)

base['CV_SG_H'] = base['DesvPad_SG_H'] / base['Media_SG_H']
base['CV_SG_A'] = base['DesvPad_SG_A'] / base['Media_SG_A']
base['SG_H_01'] = (base['Goals_H'] - base['Goals_A']) / base['p_H']
base['SG_A_01'] = (base['Goals_A'] - base['Goals_H']) / base['p_A']
base['Media_SG_H_01'] = base.groupby('Home')['SG_H_01'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['Media_SG_A_01'] = base.groupby('Away')['SG_A_01'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['Media_SG_H_01'] = base.groupby('Home')['Media_SG_H_01'].shift(1)
base['Media_SG_A_01'] = base.groupby('Away')['Media_SG_A_01'].shift(1)
base['DesvPad_SG_H_01'] = base.groupby('Home')['SG_H_01'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['DesvPad_SG_A_01'] = base.groupby('Away')['SG_A_01'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['DesvPad_SG_H_01'] = base.groupby('Home')['DesvPad_SG_H_01'].shift(1)
base['DesvPad_SG_A_01'] = base.groupby('Away')['DesvPad_SG_A_01'].shift(1)
base['CV_SG_H_01'] = base['DesvPad_SG_H_01'] / base['Media_SG_H_01']
base['CV_SG_A_01'] = base['DesvPad_SG_A_01'] / base['Media_SG_A_01']

base['SG_H_02'] = (base['Goals_H'] - base['Goals_A']) / base['p_A']
base['SG_A_02'] = (base['Goals_A'] - base['Goals_H']) / base['p_H']
base['Media_SG_H_02'] = base.groupby('Home')['SG_H_02'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['Media_SG_A_02'] = base.groupby('Away')['SG_A_02'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['Media_SG_H_02'] = base.groupby('Home')['Media_SG_H_02'].shift(1)
base['Media_SG_A_02'] = base.groupby('Away')['Media_SG_A_02'].shift(1)
base['DesvPad_SG_H_02'] = base.groupby('Home')['SG_H_02'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['DesvPad_SG_A_02'] = base.groupby('Away')['SG_A_02'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['DesvPad_SG_H_02'] = base.groupby('Home')['DesvPad_SG_H_02'].shift(1)
base['DesvPad_SG_A_02'] = base.groupby('Away')['DesvPad_SG_A_02'].shift(1)
base['CV_SG_H_02'] = base['DesvPad_SG_H_02'] / base['Media_SG_H_02']
base['CV_SG_A_02'] = base['DesvPad_SG_A_02'] / base['Media_SG_A_02']

base['VG_H'] = base['Goals_H'] * base['p_A']
base['VG_A'] = base['Goals_A'] * base['p_H']
base['Media_VG_H'] = base.groupby('Home')['VG_H'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['Media_VG_A'] = base.groupby('Away')['VG_A'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['Media_VG_H'] = base.groupby('Home')['Media_VG_H'].shift(1)
base['Media_VG_A'] = base.groupby('Away')['Media_VG_A'].shift(1)
base['DesvPad_VG_H'] = base.groupby('Home')['VG_H'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['DesvPad_VG_A'] = base.groupby('Away')['VG_A'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['DesvPad_VG_H'] = base.groupby('Home')['DesvPad_VG_H'].shift(1)
base['DesvPad_VG_A'] = base.groupby('Away')['DesvPad_VG_A'].shift(1)

base['CV_VG_H'] = base['DesvPad_VG_H'] / base['Media_VG_H']
base['CV_VG_A'] = base['DesvPad_VG_A'] / base['Media_VG_A']
base['CG_H_00'] = base['p_H'] / base['Goals_H']
base['CG_A_00'] = base['p_A'] / base['Goals_A']
base.replace(np.inf, 1, inplace=True)
base['Media_CG_H_00'] = base.groupby('Home')['CG_H_00'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['Media_CG_A_00'] = base.groupby('Away')['CG_A_00'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['Media_CG_H_00'] = base.groupby('Home')['Media_CG_H_00'].shift(1)
base['Media_CG_A_00'] = base.groupby('Away')['Media_CG_A_00'].shift(1)
base['DesvPad_CG_H_00'] = base.groupby('Home')['CG_H_00'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['DesvPad_CG_A_00'] = base.groupby('Away')['CG_A_00'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['DesvPad_CG_H_00'] = base.groupby('Home')['DesvPad_CG_H_00'].shift(1)
base['DesvPad_CG_A_00'] = base.groupby('Away')['DesvPad_CG_A_00'].shift(1)
base['CV_CG_H_00'] = base['DesvPad_CG_H_00'] / base['Media_CG_H_00']
base['CV_CG_A_00'] = base['DesvPad_CG_A_00'] / base['Media_CG_A_00']

base['CG_H_01'] = base['Goals_H'] / base['p_H']
base['CG_A_01'] = base['Goals_A'] / base['p_A']
base['Media_CG_H_01'] = base.groupby('Home')['CG_H_01'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['Media_CG_A_01'] = base.groupby('Away')['CG_A_01'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['Media_CG_H_01'] = base.groupby('Home')['Media_CG_H_01'].shift(1)
base['Media_CG_A_01'] = base.groupby('Away')['Media_CG_A_01'].shift(1)
base['DesvPad_CG_H_01'] = base.groupby('Home')['CG_H_01'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['DesvPad_CG_A_01'] = base.groupby('Away')['CG_A_01'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['DesvPad_CG_H_01'] = base.groupby('Home')['DesvPad_CG_H_01'].shift(1)
base['DesvPad_CG_A_01'] = base.groupby('Away')['DesvPad_CG_A_01'].shift(1)
base['CV_CG_H_01'] = base['DesvPad_CG_H_01'] / base['Media_CG_H_01']
base['CV_CG_A_01'] = base['DesvPad_CG_A_01'] / base['Media_CG_A_01']

base['CG_H_02'] = (base['Goals_H'] / 2) + (base['p_H'] / 2)
base['CG_A_02'] = (base['Goals_A'] / 2) + (base['p_A'] / 2)
base['Media_CG_H_02'] = base.groupby('Home')['CG_H_02'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['Media_CG_A_02'] = base.groupby('Away')['CG_A_02'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['Media_CG_H_02'] = base.groupby('Home')['Media_CG_H_02'].shift(1)
base['Media_CG_A_02'] = base.groupby('Away')['Media_CG_A_02'].shift(1)
base['DesvPad_CG_H_02'] = base.groupby('Home')['CG_H_02'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['DesvPad_CG_A_02'] = base.groupby('Away')['CG_A_02'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['DesvPad_CG_H_02'] = base.groupby('Home')['DesvPad_CG_H_02'].shift(1)
base['DesvPad_CG_A_02'] = base.groupby('Away')['DesvPad_CG_A_02'].shift(1)
base['CV_CG_H_02'] = base['DesvPad_CG_H_02'] / base['Media_CG_H_02']
base['CV_CG_A_02'] = base['DesvPad_CG_A_02'] / base['Media_CG_A_02']

base['H_D'] = base['Odd_H'] / base['Odd_D']
base['H_A'] = base['Odd_H'] / base['Odd_A']
base['D_H'] = base['Odd_D'] / base['Odd_H']
base['D_A'] = base['Odd_D'] / base['Odd_A']
base['A_H'] = base['Odd_A'] / base['Odd_H']
base['A_D'] = base['Odd_A'] / base['Odd_D']

base['DifAbs_HomeAway'] = np.abs(base['Odd_H'] - base['Odd_A'])
base['DifAbs_HomeDraw'] = np.abs(base['Odd_H'] - base['Odd_D'])
base['DifAbs_DrawAway'] = np.abs(base['Odd_D'] - base['Odd_A'])

base['Angle_HomeAway'] = np.degrees(np.arctan((base['Odd_A'] - base['Odd_H']) / 2))
base['Angle_HomeDraw'] = np.degrees(np.arctan((base['Odd_D'] - base['Odd_H']) / 2))
base['Angle_DrawAway'] = np.degrees(np.arctan((base['Odd_A'] - base['Odd_D']) / 2))

base['DifPer_HomeAway'] = np.abs((base['Odd_H'] - base['Odd_A'])) / base['Odd_A']
base['DifPer_HomeDraw'] = np.abs((base['Odd_H'] - base['Odd_D'])) / base['Odd_D']
base['DifPer_DrawAway'] = np.abs((base['Odd_D'] - base['Odd_A'])) / base['Odd_A']

colunas_deletar = ['Ptos_H','Ptos_A','DesvPad_Ptos_H','DesvPad_Ptos_A',
                   'DesvPad_GM_H','DesvPad_GM_A',
                   'DesvPad_GS_H','DesvPad_GS_A',
                   'SG_H','SG_A','DesvPad_SG_H','DesvPad_SG_A',
                   'SG_H_01','SG_A_01','DesvPad_SG_H_01','DesvPad_SG_A_01',
                   'SG_H_02','SG_A_02','DesvPad_SG_H_02','DesvPad_SG_A_02',
                   'VG_H','VG_A','DesvPad_VG_H','DesvPad_VG_A',
                   'CG_H_00','CG_A_00','DesvPad_CG_H_00','DesvPad_CG_A_00',
                   'CG_H_01','CG_A_01','DesvPad_CG_H_01','DesvPad_CG_A_01',
                   'CG_H_02','CG_A_02','DesvPad_CG_H_02','DesvPad_CG_A_02']

base = base.drop(colunas_deletar, axis=1)
base = drop_reset_index(base)

In [ ]:
base.loc[(base['Goals_H'] >  base['Goals_A']), 'Back_Home'] = 1
base.loc[(base['Goals_H'] <= base['Goals_A']), 'Back_Home'] = 0

stake = 1
green = stake * (base.Odd_H - 1)
red = -stake

base.loc[(base['Back_Home'] == 1), 'Profit'] = green
base.loc[(base['Back_Home'] == 0), 'Profit'] = red

In [ ]:
treino = base[base['Date'].dt.year <= 2022]
teste = base[base['Date'].dt.year >= 2023]

features = [
 'p_H',
 'Angle_HomeAway',
 'Angle_DrawAway',
 'Angle_HomeDraw',
 'A_D',
 'D_H',
 'A_H',
 'DifPer_HomeDraw',
 'DifAbs_HomeDraw',
 'CV_Odds',
 'DifAbs_HomeAway',
 'Media_CG_H_02',
 'Media_SG_H',
 'Media_SG_H_02',
 'Media_Ptos_H',
 'Media_GM_H',
 'Media_SG_H_01',
 'CV_Ptos_A',
 'Media_GS_A',
 'DifAbs_DrawAway',
 'CV_CG_A_02',
 'CV_CG_A_01',
 'CV_GS_H',
 'CV_CG_A_00',
 'CV_SG_H',
 'CV_GM_A',
 'Media_CG_A_01',
 'CV_VG_A',
 'Media_VG_A',
 'Media_CG_A_00',
 'CV_SG_H_02',
 'CV_SG_H_01',
 'CV_SG_A_02',
 'CV_SG_A_01',
 'Media_CG_H_01',
 'Media_VG_H',
 'Media_CG_H_00',
 'CV_CG_H_00',
 'CV_SG_A',
 'CV_VG_H',
 'CV_GS_A',
 'CV_GM_H',
 'CV_CG_H_02',
 'CV_CG_H_01',
 'Media_GM_A',
 'Media_GS_H',
 'DifPer_HomeAway',
 'Media_SG_A_02',
 'Media_CG_A_02',
 'Media_Ptos_A',
 'Media_SG_A_01',
 'CV_Ptos_H',
 'Media_SG_A',
 'p_D',
 'DifPer_DrawAway',
 'H_A',
 'D_A',
 'H_D',
 'p_A'
]

label = 'Back_Home'

Back_Home = setup(data = treino[features], target = treino[label], session_id=123, verbose = False)

In [ ]:
df_models = models()
modelos = df_models.index.tolist()
modelos

In [ ]:
lista = []

for i in modelos:
    modelo = create_model(i, verbose=False)
    df0 = predict_model(modelo, teste, verbose=False)

    try:
        filtro = df0.prediction_label == 1
        df0 = df0[filtro]
        df0 = drop_reset_index(df0)

        df0['Profit_acu'] = df0.Profit.cumsum()
        n_apostas = df0.shape[0]
        profit = round(df0.Profit_acu.tail(1).item(), 2)
        ROI = round((df0.Profit_acu.tail(1) / n_apostas * 100).item(), 2)
        drawdown = df0['Profit_acu'] - df0['Profit_acu'].cummax()
        drawdown_maximo = round(drawdown.min(), 2)
        winrate_medio = round((df0['Profit'] > 0).mean() * 100, 2)
        desvio_padrao = round(df0['Profit'].std(), 2)

        metodos = {}

        metodos["Método"] = i
        metodos["Profit"] = profit
        metodos["Nº de Jogos"] = n_apostas
        metodos["ROI"] = ROI
        metodos["DrawDown"] = drawdown_maximo
        metodos["WinRate"] = winrate_medio
        metodos["DesvPad"] = desvio_padrao

        lista.append(metodos)
        print(f'Modelo: {i}, Profit: {profit}')

    except:
        print(f'Erro no Modelo {modelo}')

In [ ]:
df_modelos = pd.DataFrame(lista)
df_modelos = df_modelos.sort_values(by='ROI',ascending=False)
flt = (df_modelos['Nº de Jogos'] >= 100) & (df_modelos['ROI'] >= 5)
df_modelos = df_modelos[flt]
df_modelos = drop_reset_index(df_modelos)
display(df_modelos.head())

In [ ]:
base = base[base['League'].isin(leagues) == True]
base = drop_reset_index(base)

n_per = 5

base['p_H'] = 1 / base['Odd_H']
base['p_D'] = 1 / base['Odd_D']
base['p_A'] = 1 / base['Odd_A']

desvio_padrao = base[['Odd_H','Odd_D','Odd_A']].std(axis=1)
media = base[['Odd_H','Odd_D','Odd_A']].mean(axis=1)
CV_Odds = desvio_padrao / media
base['CV_Odds'] = CV_Odds

base['Ptos_H'] = np.where(base['Goals_H'] >  base['Goals_A'], 3,
               np.where(base['Goals_H'] == base['Goals_A'], 1, 0))
base['Ptos_A'] = np.where(base['Goals_H'] > base['Goals_A'], 0,
               np.where(base['Goals_H'] == base['Goals_A'], 1, 3))

base['Media_Ptos_H'] = base.groupby('Home')['Ptos_H'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['Media_Ptos_A'] = base.groupby('Away')['Ptos_A'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['DesvPad_Ptos_H'] = base.groupby('Home')['Ptos_H'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['DesvPad_Ptos_A'] = base.groupby('Away')['Ptos_A'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['CV_Ptos_H'] = base['DesvPad_Ptos_H'] / base['Media_Ptos_H']
base['CV_Ptos_A'] = base['DesvPad_Ptos_A'] / base['Media_Ptos_A']

base['Media_GM_H'] = base.groupby('Home')['Goals_H'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['Media_GM_A'] = base.groupby('Away')['Goals_A'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['DesvPad_GM_H'] = base.groupby('Home')['Goals_H'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['DesvPad_GM_A'] = base.groupby('Away')['Goals_A'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['CV_GM_H'] = base['DesvPad_GM_H'] / base['Media_GM_H']
base['CV_GM_A'] = base['DesvPad_GM_A'] / base['Media_GM_A']

base['Media_GS_H'] = base.groupby('Home')['Goals_A'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['Media_GS_A'] = base.groupby('Away')['Goals_H'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['DesvPad_GS_H'] = base.groupby('Home')['Goals_A'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['DesvPad_GS_A'] = base.groupby('Away')['Goals_H'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['CV_GS_H'] = base['DesvPad_GS_H'] / base['Media_GS_H']
base['CV_GS_A'] = base['DesvPad_GS_A'] / base['Media_GS_A']

base['SG_H'] = base['Goals_H'] - base['Goals_A']
base['SG_A'] = base['Goals_A'] - base['Goals_H']
base['Media_SG_H'] = base.groupby('Home')['SG_H'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['Media_SG_A'] = base.groupby('Away')['SG_A'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['DesvPad_SG_H'] = base.groupby('Home')['SG_H'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['DesvPad_SG_A'] = base.groupby('Away')['SG_A'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['CV_SG_H'] = base['DesvPad_SG_H'] / base['Media_SG_H']
base['CV_SG_A'] = base['DesvPad_SG_A'] / base['Media_SG_A']

base['SG_H_01'] = (base['Goals_H'] - base['Goals_A']) / base['p_H']
base['SG_A_01'] = (base['Goals_A'] - base['Goals_H']) / base['p_A']
base['Media_SG_H_01'] = base.groupby('Home')['SG_H_01'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['Media_SG_A_01'] = base.groupby('Away')['SG_A_01'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['DesvPad_SG_H_01'] = base.groupby('Home')['SG_H_01'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['DesvPad_SG_A_01'] = base.groupby('Away')['SG_A_01'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['CV_SG_H_01'] = base['DesvPad_SG_H_01'] / base['Media_SG_H_01']
base['CV_SG_A_01'] = base['DesvPad_SG_A_01'] / base['Media_SG_A_01']

base['SG_H_02'] = (base['Goals_H'] - base['Goals_A']) / base['p_A']
base['SG_A_02'] = (base['Goals_A'] - base['Goals_H']) / base['p_H']
base['Media_SG_H_02'] = base.groupby('Home')['SG_H_02'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['Media_SG_A_02'] = base.groupby('Away')['SG_A_02'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['DesvPad_SG_H_02'] = base.groupby('Home')['SG_H_02'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['DesvPad_SG_A_02'] = base.groupby('Away')['SG_A_02'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['CV_SG_H_02'] = base['DesvPad_SG_H_02'] / base['Media_SG_H_02']
base['CV_SG_A_02'] = base['DesvPad_SG_A_02'] / base['Media_SG_A_02']

base['VG_H'] = base['Goals_H'] * base['p_A']
base['VG_A'] = base['Goals_A'] * base['p_H']
base['Media_VG_H'] = base.groupby('Home')['VG_H'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['Media_VG_A'] = base.groupby('Away')['VG_A'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['DesvPad_VG_H'] = base.groupby('Home')['VG_H'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['DesvPad_VG_A'] = base.groupby('Away')['VG_A'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['CV_VG_H'] = base['DesvPad_VG_H'] / base['Media_VG_H']
base['CV_VG_A'] = base['DesvPad_VG_A'] / base['Media_VG_A']

base['CG_H_00'] = base['p_H'] / base['Goals_H']
base['CG_A_00'] = base['p_A'] / base['Goals_A']
base.replace(np.inf, 1, inplace=True)
base['Media_CG_H_00'] = base.groupby('Home')['CG_H_00'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['Media_CG_A_00'] = base.groupby('Away')['CG_A_00'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['DesvPad_CG_H_00'] = base.groupby('Home')['CG_H_00'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['DesvPad_CG_A_00'] = base.groupby('Away')['CG_A_00'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['CV_CG_H_00'] = base['DesvPad_CG_H_00'] / base['Media_CG_H_00']
base['CV_CG_A_00'] = base['DesvPad_CG_A_00'] / base['Media_CG_A_00']

base['CG_H_01'] = base['Goals_H'] / base['p_H']
base['CG_A_01'] = base['Goals_A'] / base['p_A']
base['Media_CG_H_01'] = base.groupby('Home')['CG_H_01'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['Media_CG_A_01'] = base.groupby('Away')['CG_A_01'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['DesvPad_CG_H_01'] = base.groupby('Home')['CG_H_01'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['DesvPad_CG_A_01'] = base.groupby('Away')['CG_A_01'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['CV_CG_H_01'] = base['DesvPad_CG_H_01'] / base['Media_CG_H_01']
base['CV_CG_A_01'] = base['DesvPad_CG_A_01'] / base['Media_CG_A_01']

base['CG_H_02'] = (base['Goals_H'] / 2) + (base['p_H'] / 2)
base['CG_A_02'] = (base['Goals_A'] / 2) + (base['p_A'] / 2)
base['Media_CG_H_02'] = base.groupby('Home')['CG_H_02'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['Media_CG_A_02'] = base.groupby('Away')['CG_A_02'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
base['DesvPad_CG_H_02'] = base.groupby('Home')['CG_H_02'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['DesvPad_CG_A_02'] = base.groupby('Away')['CG_A_02'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
base['CV_CG_H_02'] = base['DesvPad_CG_H_02'] / base['Media_CG_H_02']
base['CV_CG_A_02'] = base['DesvPad_CG_A_02'] / base['Media_CG_A_02']

base['H_D'] = base['Odd_H'] / base['Odd_D']
base['H_A'] = base['Odd_H'] / base['Odd_A']
base['D_H'] = base['Odd_D'] / base['Odd_H']
base['D_A'] = base['Odd_D'] / base['Odd_A']
base['A_H'] = base['Odd_A'] / base['Odd_H']
base['A_D'] = base['Odd_A'] / base['Odd_D']

base['DifAbs_HomeAway'] = np.abs(base['Odd_H'] - base['Odd_A'])
base['DifAbs_HomeDraw'] = np.abs(base['Odd_H'] - base['Odd_D'])
base['DifAbs_DrawAway'] = np.abs(base['Odd_D'] - base['Odd_A'])

base['Angle_HomeAway'] = np.degrees(np.arctan((base['Odd_A'] - base['Odd_H']) / 2))
base['Angle_HomeDraw'] = np.degrees(np.arctan((base['Odd_D'] - base['Odd_H']) / 2))
base['Angle_DrawAway'] = np.degrees(np.arctan((base['Odd_A'] - base['Odd_D']) / 2))

base['DifPer_HomeAway'] = np.abs((base['Odd_H'] - base['Odd_A'])) / base['Odd_A']
base['DifPer_HomeDraw'] = np.abs((base['Odd_H'] - base['Odd_D'])) / base['Odd_D']
base['DifPer_DrawAway'] = np.abs((base['Odd_D'] - base['Odd_A'])) / base['Odd_A']

colunas_deletar = ['Ptos_H','Ptos_A','DesvPad_Ptos_H','DesvPad_Ptos_A',
                   'DesvPad_GM_H','DesvPad_GM_A',
                   'DesvPad_GS_H','DesvPad_GS_A',
                   'SG_H','SG_A','DesvPad_SG_H','DesvPad_SG_A',
                   'SG_H_01','SG_A_01','DesvPad_SG_H_01','DesvPad_SG_A_01',
                   'SG_H_02','SG_A_02','DesvPad_SG_H_02','DesvPad_SG_A_02',
                   'VG_H','VG_A','DesvPad_VG_H','DesvPad_VG_A',
                   'CG_H_00','CG_A_00','DesvPad_CG_H_00','DesvPad_CG_A_00',
                   'CG_H_01','CG_A_01','DesvPad_CG_H_01','DesvPad_CG_A_01',
                   'CG_H_02','CG_A_02','DesvPad_CG_H_02','DesvPad_CG_A_02']

base = base.drop(colunas_deletar, axis=1)
base = drop_reset_index(base)

In [ ]:
best_method = df_modelos['Método'][1]
Back_Home_PyCaret = create_model(best_method, verbose=False)

jogos_do_dia = pd.read_csv('https://github.com/futpythontrader/YouTube/blob/main/Jogos_do_Dia_FlashScore/'+str(dia)+'_Jogos_do_Dia_FlashScore.csv?raw=true')
jogos_do_dia = jogos_do_dia[['League','Date','Time','Home','Away','Odd_H','Odd_D','Odd_A']]
Jogos_do_Dia = jogos_do_dia[jogos_do_dia['League'].isin(leagues) == True]
Jogos_do_Dia = drop_reset_index(Jogos_do_Dia)

Jogos_do_Dia['p_H'] = 1 / Jogos_do_Dia['Odd_H']
Jogos_do_Dia['p_D'] = 1 / Jogos_do_Dia['Odd_D']
Jogos_do_Dia['p_A'] = 1 / Jogos_do_Dia['Odd_A']

desvio_padrao = Jogos_do_Dia[['Odd_H','Odd_D','Odd_A']].std(axis=1)
media = Jogos_do_Dia[['Odd_H','Odd_D','Odd_A']].mean(axis=1)
CV_Odds = desvio_padrao / media
Jogos_do_Dia['CV_Odds'] = CV_Odds

Jogos_do_Dia['H_D'] = Jogos_do_Dia['Odd_H'] / Jogos_do_Dia['Odd_D']
Jogos_do_Dia['H_A'] = Jogos_do_Dia['Odd_H'] / Jogos_do_Dia['Odd_A']
Jogos_do_Dia['D_H'] = Jogos_do_Dia['Odd_D'] / Jogos_do_Dia['Odd_H']
Jogos_do_Dia['D_A'] = Jogos_do_Dia['Odd_D'] / Jogos_do_Dia['Odd_A']
Jogos_do_Dia['A_H'] = Jogos_do_Dia['Odd_A'] / Jogos_do_Dia['Odd_H']
Jogos_do_Dia['A_D'] = Jogos_do_Dia['Odd_A'] / Jogos_do_Dia['Odd_D']

Jogos_do_Dia['DifAbs_HomeAway'] = np.abs(Jogos_do_Dia['Odd_H'] - Jogos_do_Dia['Odd_A'])
Jogos_do_Dia['DifAbs_HomeDraw'] = np.abs(Jogos_do_Dia['Odd_H'] - Jogos_do_Dia['Odd_D'])
Jogos_do_Dia['DifAbs_DrawAway'] = np.abs(Jogos_do_Dia['Odd_D'] - Jogos_do_Dia['Odd_A'])

Jogos_do_Dia['Angle_HomeAway'] = np.degrees(np.arctan((Jogos_do_Dia['Odd_A'] - Jogos_do_Dia['Odd_H']) / 2))
Jogos_do_Dia['Angle_HomeDraw'] = np.degrees(np.arctan((Jogos_do_Dia['Odd_D'] - Jogos_do_Dia['Odd_H']) / 2))
Jogos_do_Dia['Angle_DrawAway'] = np.degrees(np.arctan((Jogos_do_Dia['Odd_A'] - Jogos_do_Dia['Odd_D']) / 2))

Jogos_do_Dia['DifPer_HomeAway'] = np.abs((Jogos_do_Dia['Odd_H'] - Jogos_do_Dia['Odd_A'])) / Jogos_do_Dia['Odd_A']
Jogos_do_Dia['DifPer_HomeDraw'] = np.abs((Jogos_do_Dia['Odd_H'] - Jogos_do_Dia['Odd_D'])) / Jogos_do_Dia['Odd_D']
Jogos_do_Dia['DifPer_DrawAway'] = np.abs((Jogos_do_Dia['Odd_D'] - Jogos_do_Dia['Odd_A'])) / Jogos_do_Dia['Odd_A']

base_H = base[['Home',
                'Media_Ptos_H','CV_Ptos_H',
                'Media_GM_H','CV_GM_H',
                'Media_GS_H','CV_GS_H',
                'Media_SG_H','CV_SG_H',
                'Media_SG_H_01','CV_SG_H_01',
                'Media_SG_H_02','CV_SG_H_02',
                'Media_VG_H','CV_VG_H',
                'Media_CG_H_00','CV_CG_H_00',
                'Media_CG_H_01','CV_CG_H_01',
                'Media_CG_H_02','CV_CG_H_02']]

base_A = base[['Away',
                'Media_Ptos_A','CV_Ptos_A',
                'Media_GM_A','CV_GM_A',
                'Media_GS_A','CV_GS_A',
                'Media_SG_A','CV_SG_A',
                'Media_SG_A_01','CV_SG_A_01',
                'Media_SG_A_02','CV_SG_A_02',
                'Media_VG_A','CV_VG_A',
                'Media_CG_A_00','CV_CG_A_00',
                'Media_CG_A_01','CV_CG_A_01',
                'Media_CG_A_02','CV_CG_A_02']]

ultima_base_H = base_H.groupby('Home').last().reset_index()
ultima_base_A = base_A.groupby('Away').last().reset_index()

df = pd.merge(Jogos_do_Dia, ultima_base_H, how='left', left_on='Home', right_on='Home')
df = pd.merge(df, ultima_base_A, how='left', left_on='Away', right_on='Away')

df = drop_reset_index(df)

df1 = predict_model(Back_Home_PyCaret, data=df)
filtro = df1.prediction_label == 1
Entradas = df1[filtro]
Entradas = Entradas[['League','Date','Time','Home','Away','Odd_H','Odd_D','Odd_A']]
Entradas = drop_reset_index(Entradas)
display(Entradas)


### Rodando com Todas as Ligas Selecionadas

In [ ]:
Jogos_do_Dia = read_jogos(dia)
not_leagues = ['SOUTH AFRICA - PREMIER LEAGUE']
Jogos_do_Dia = Jogos_do_Dia[Jogos_do_Dia['League'].isin(not_leagues) == False]

Jogos_do_Dia = Jogos_do_Dia.sort_values(by='League')
leagues = Jogos_do_Dia['League'].unique()

if len(leagues != 0):
    df_Entradas = pd.DataFrame()
    df_Modelos = pd.DataFrame()

    for liga in leagues:

        print('');print(liga);print('')

        lista_ligas = [liga]

        flt = base_orig.Date < str(dia)
        base = base_orig[flt]
        base = base[base['League'].isin(lista_ligas) == True]
        base = drop_reset_index(base)

        n_per = 5

        base['p_H'] = 1 / base['Odd_H']
        base['p_D'] = 1 / base['Odd_D']
        base['p_A'] = 1 / base['Odd_A']

        desvio_padrao = base[['Odd_H','Odd_D','Odd_A']].std(axis=1)
        media = base[['Odd_H','Odd_D','Odd_A']].mean(axis=1)
        CV_Odds = desvio_padrao / media
        base['CV_Odds'] = CV_Odds

        base['Ptos_H'] = np.where(base['Goals_H'] >  base['Goals_A'], 3,
                    np.where(base['Goals_H'] == base['Goals_A'], 1, 0))
        base['Ptos_A'] = np.where(base['Goals_H'] > base['Goals_A'], 0,
                    np.where(base['Goals_H'] == base['Goals_A'], 1, 3))
        base['Media_Ptos_H'] = base.groupby('Home')['Ptos_H'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
        base['Media_Ptos_A'] = base.groupby('Away')['Ptos_A'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
        base['Media_Ptos_H'] = base.groupby('Home')['Media_Ptos_H'].shift(1)
        base['Media_Ptos_A'] = base.groupby('Away')['Media_Ptos_A'].shift(1)
        base['DesvPad_Ptos_H'] = base.groupby('Home')['Ptos_H'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
        base['DesvPad_Ptos_A'] = base.groupby('Away')['Ptos_A'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
        base['DesvPad_Ptos_H'] = base.groupby('Home')['DesvPad_Ptos_H'].shift(1)
        base['DesvPad_Ptos_A'] = base.groupby('Away')['DesvPad_Ptos_A'].shift(1)
        base['CV_Ptos_H'] = base['DesvPad_Ptos_H'] / base['Media_Ptos_H']
        base['CV_Ptos_A'] = base['DesvPad_Ptos_A'] / base['Media_Ptos_A']

        base['Media_GM_H'] = base.groupby('Home')['Goals_H'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
        base['Media_GM_A'] = base.groupby('Away')['Goals_A'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
        base['Media_GM_H'] = base.groupby('Home')['Media_GM_H'].shift(1)
        base['Media_GM_A'] = base.groupby('Away')['Media_GM_A'].shift(1)
        base['DesvPad_GM_H'] = base.groupby('Home')['Goals_H'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
        base['DesvPad_GM_A'] = base.groupby('Away')['Goals_A'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
        base['DesvPad_GM_H'] = base.groupby('Home')['DesvPad_GM_H'].shift(1)
        base['DesvPad_GM_A'] = base.groupby('Away')['DesvPad_GM_A'].shift(1)
        base['CV_GM_H'] = base['DesvPad_GM_H'] / base['Media_GM_H']
        base['CV_GM_A'] = base['DesvPad_GM_A'] / base['Media_GM_A']

        base['Media_GS_H'] = base.groupby('Home')['Goals_A'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
        base['Media_GS_A'] = base.groupby('Away')['Goals_H'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
        base['Media_GS_H'] = base.groupby('Home')['Media_GS_H'].shift(1)
        base['Media_GS_A'] = base.groupby('Away')['Media_GS_A'].shift(1)
        base['DesvPad_GS_H'] = base.groupby('Home')['Goals_A'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
        base['DesvPad_GS_A'] = base.groupby('Away')['Goals_H'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
        base['DesvPad_GS_H'] = base.groupby('Home')['DesvPad_GS_H'].shift(1)
        base['DesvPad_GS_A'] = base.groupby('Away')['DesvPad_GS_A'].shift(1)
        base['CV_GS_H'] = base['DesvPad_GS_H'] / base['Media_GS_H']
        base['CV_GS_A'] = base['DesvPad_GS_A'] / base['Media_GS_A']

        base['SG_H'] = base['Goals_H'] - base['Goals_A']
        base['SG_A'] = base['Goals_A'] - base['Goals_H']
        base['Media_SG_H'] = base.groupby('Home')['SG_H'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
        base['Media_SG_A'] = base.groupby('Away')['SG_A'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
        base['Media_SG_H'] = base.groupby('Home')['Media_SG_H'].shift(1)
        base['Media_SG_A'] = base.groupby('Away')['Media_SG_A'].shift(1)
        base['DesvPad_SG_H'] = base.groupby('Home')['SG_H'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
        base['DesvPad_SG_A'] = base.groupby('Away')['SG_A'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
        base['DesvPad_SG_H'] = base.groupby('Home')['DesvPad_SG_H'].shift(1)
        base['DesvPad_SG_A'] = base.groupby('Away')['DesvPad_SG_A'].shift(1)
        base['CV_SG_H'] = base['DesvPad_SG_H'] / base['Media_SG_H']
        base['CV_SG_A'] = base['DesvPad_SG_A'] / base['Media_SG_A']

        base['SG_H_01'] = (base['Goals_H'] - base['Goals_A']) / base['p_H']
        base['SG_A_01'] = (base['Goals_A'] - base['Goals_H']) / base['p_A']
        base['Media_SG_H_01'] = base.groupby('Home')['SG_H_01'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
        base['Media_SG_A_01'] = base.groupby('Away')['SG_A_01'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
        base['Media_SG_H_01'] = base.groupby('Home')['Media_SG_H_01'].shift(1)
        base['Media_SG_A_01'] = base.groupby('Away')['Media_SG_A_01'].shift(1)
        base['DesvPad_SG_H_01'] = base.groupby('Home')['SG_H_01'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
        base['DesvPad_SG_A_01'] = base.groupby('Away')['SG_A_01'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
        base['DesvPad_SG_H_01'] = base.groupby('Home')['DesvPad_SG_H_01'].shift(1)
        base['DesvPad_SG_A_01'] = base.groupby('Away')['DesvPad_SG_A_01'].shift(1)
        base['CV_SG_H_01'] = base['DesvPad_SG_H_01'] / base['Media_SG_H_01']
        base['CV_SG_A_01'] = base['DesvPad_SG_A_01'] / base['Media_SG_A_01']

        base['SG_H_02'] = (base['Goals_H'] - base['Goals_A']) / base['p_A']
        base['SG_A_02'] = (base['Goals_A'] - base['Goals_H']) / base['p_H']
        base['Media_SG_H_02'] = base.groupby('Home')['SG_H_02'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
        base['Media_SG_A_02'] = base.groupby('Away')['SG_A_02'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
        base['Media_SG_H_02'] = base.groupby('Home')['Media_SG_H_02'].shift(1)
        base['Media_SG_A_02'] = base.groupby('Away')['Media_SG_A_02'].shift(1)
        base['DesvPad_SG_H_02'] = base.groupby('Home')['SG_H_02'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
        base['DesvPad_SG_A_02'] = base.groupby('Away')['SG_A_02'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
        base['DesvPad_SG_H_02'] = base.groupby('Home')['DesvPad_SG_H_02'].shift(1)
        base['DesvPad_SG_A_02'] = base.groupby('Away')['DesvPad_SG_A_02'].shift(1)
        base['CV_SG_H_02'] = base['DesvPad_SG_H_02'] / base['Media_SG_H_02']
        base['CV_SG_A_02'] = base['DesvPad_SG_A_02'] / base['Media_SG_A_02']

        base['VG_H'] = base['Goals_H'] * base['p_A']
        base['VG_A'] = base['Goals_A'] * base['p_H']
        base['Media_VG_H'] = base.groupby('Home')['VG_H'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
        base['Media_VG_A'] = base.groupby('Away')['VG_A'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
        base['Media_VG_H'] = base.groupby('Home')['Media_VG_H'].shift(1)
        base['Media_VG_A'] = base.groupby('Away')['Media_VG_A'].shift(1)
        base['DesvPad_VG_H'] = base.groupby('Home')['VG_H'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
        base['DesvPad_VG_A'] = base.groupby('Away')['VG_A'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
        base['DesvPad_VG_H'] = base.groupby('Home')['DesvPad_VG_H'].shift(1)
        base['DesvPad_VG_A'] = base.groupby('Away')['DesvPad_VG_A'].shift(1)
        base['CV_VG_H'] = base['DesvPad_VG_H'] / base['Media_VG_H']
        base['CV_VG_A'] = base['DesvPad_VG_A'] / base['Media_VG_A']

        base['CG_H_00'] = base['p_H'] / base['Goals_H']
        base['CG_A_00'] = base['p_A'] / base['Goals_A']
        base.replace(np.inf, 1, inplace=True)
        base['Media_CG_H_00'] = base.groupby('Home')['CG_H_00'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
        base['Media_CG_A_00'] = base.groupby('Away')['CG_A_00'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
        base['Media_CG_H_00'] = base.groupby('Home')['Media_CG_H_00'].shift(1)
        base['Media_CG_A_00'] = base.groupby('Away')['Media_CG_A_00'].shift(1)
        base['DesvPad_CG_H_00'] = base.groupby('Home')['CG_H_00'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
        base['DesvPad_CG_A_00'] = base.groupby('Away')['CG_A_00'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
        base['DesvPad_CG_H_00'] = base.groupby('Home')['DesvPad_CG_H_00'].shift(1)
        base['DesvPad_CG_A_00'] = base.groupby('Away')['DesvPad_CG_A_00'].shift(1)
        base['CV_CG_H_00'] = base['DesvPad_CG_H_00'] / base['Media_CG_H_00']
        base['CV_CG_A_00'] = base['DesvPad_CG_A_00'] / base['Media_CG_A_00']

        base['CG_H_01'] = base['Goals_H'] / base['p_H']
        base['CG_A_01'] = base['Goals_A'] / base['p_A']
        base['Media_CG_H_01'] = base.groupby('Home')['CG_H_01'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
        base['Media_CG_A_01'] = base.groupby('Away')['CG_A_01'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
        base['Media_CG_H_01'] = base.groupby('Home')['Media_CG_H_01'].shift(1)
        base['Media_CG_A_01'] = base.groupby('Away')['Media_CG_A_01'].shift(1)
        base['DesvPad_CG_H_01'] = base.groupby('Home')['CG_H_01'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
        base['DesvPad_CG_A_01'] = base.groupby('Away')['CG_A_01'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
        base['DesvPad_CG_H_01'] = base.groupby('Home')['DesvPad_CG_H_01'].shift(1)
        base['DesvPad_CG_A_01'] = base.groupby('Away')['DesvPad_CG_A_01'].shift(1)
        base['CV_CG_H_01'] = base['DesvPad_CG_H_01'] / base['Media_CG_H_01']
        base['CV_CG_A_01'] = base['DesvPad_CG_A_01'] / base['Media_CG_A_01']

        base['CG_H_02'] = (base['Goals_H'] / 2) + (base['p_H'] / 2)
        base['CG_A_02'] = (base['Goals_A'] / 2) + (base['p_A'] / 2)
        base['Media_CG_H_02'] = base.groupby('Home')['CG_H_02'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
        base['Media_CG_A_02'] = base.groupby('Away')['CG_A_02'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
        base['Media_CG_H_02'] = base.groupby('Home')['Media_CG_H_02'].shift(1)
        base['Media_CG_A_02'] = base.groupby('Away')['Media_CG_A_02'].shift(1)
        base['DesvPad_CG_H_02'] = base.groupby('Home')['CG_H_02'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
        base['DesvPad_CG_A_02'] = base.groupby('Away')['CG_A_02'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
        base['DesvPad_CG_H_02'] = base.groupby('Home')['DesvPad_CG_H_02'].shift(1)
        base['DesvPad_CG_A_02'] = base.groupby('Away')['DesvPad_CG_A_02'].shift(1)
        base['CV_CG_H_02'] = base['DesvPad_CG_H_02'] / base['Media_CG_H_02']
        base['CV_CG_A_02'] = base['DesvPad_CG_A_02'] / base['Media_CG_A_02']

        base['H_D'] = base['Odd_H'] / base['Odd_D']
        base['H_A'] = base['Odd_H'] / base['Odd_A']
        base['D_H'] = base['Odd_D'] / base['Odd_H']
        base['D_A'] = base['Odd_D'] / base['Odd_A']
        base['A_H'] = base['Odd_A'] / base['Odd_H']
        base['A_D'] = base['Odd_A'] / base['Odd_D']

        base['DifAbs_HomeAway'] = np.abs(base['Odd_H'] - base['Odd_A'])
        base['DifAbs_HomeDraw'] = np.abs(base['Odd_H'] - base['Odd_D'])
        base['DifAbs_DrawAway'] = np.abs(base['Odd_D'] - base['Odd_A'])

        base['Angle_HomeAway'] = np.degrees(np.arctan((base['Odd_A'] - base['Odd_H']) / 2))
        base['Angle_HomeDraw'] = np.degrees(np.arctan((base['Odd_D'] - base['Odd_H']) / 2))
        base['Angle_DrawAway'] = np.degrees(np.arctan((base['Odd_A'] - base['Odd_D']) / 2))

        base['DifPer_HomeAway'] = np.abs((base['Odd_H'] - base['Odd_A'])) / base['Odd_A']
        base['DifPer_HomeDraw'] = np.abs((base['Odd_H'] - base['Odd_D'])) / base['Odd_D']
        base['DifPer_DrawAway'] = np.abs((base['Odd_D'] - base['Odd_A'])) / base['Odd_A']

        colunas_deletar = ['Ptos_H','Ptos_A','DesvPad_Ptos_H','DesvPad_Ptos_A',
                           'DesvPad_GM_H','DesvPad_GM_A',
                           'DesvPad_GS_H','DesvPad_GS_A',
                           'SG_H','SG_A','DesvPad_SG_H','DesvPad_SG_A',
                           'SG_H_01','SG_A_01','DesvPad_SG_H_01','DesvPad_SG_A_01',
                           'SG_H_02','SG_A_02','DesvPad_SG_H_02','DesvPad_SG_A_02',
                           'VG_H','VG_A','DesvPad_VG_H','DesvPad_VG_A',
                           'CG_H_00','CG_A_00','DesvPad_CG_H_00','DesvPad_CG_A_00',
                           'CG_H_01','CG_A_01','DesvPad_CG_H_01','DesvPad_CG_A_01',
                           'CG_H_02','CG_A_02','DesvPad_CG_H_02','DesvPad_CG_A_02']

        base = base.drop(colunas_deletar, axis=1)
        base = drop_reset_index(base)

        base.loc[(base['Goals_H'] >  base['Goals_A']), 'Back_Home'] = 1
        base.loc[(base['Goals_H'] <= base['Goals_A']), 'Back_Home'] = 0

        stake = 1
        green = stake * (base.Odd_H - 1)
        red = -stake

        base.loc[(base['Back_Home'] == 1), 'Profit'] = green
        base.loc[(base['Back_Home'] == 0), 'Profit'] = red

        treino = base[base['Date'].dt.year <= 2022]
        teste = base[base['Date'].dt.year >= 2023]

        features = [
 'p_H',
 'Angle_HomeAway',
 'Angle_DrawAway',
 'Angle_HomeDraw',
 'A_D',
 'D_H',
 'A_H',
 'DifPer_HomeDraw',
 'DifAbs_HomeDraw',
 'CV_Odds',
 'DifAbs_HomeAway',
 'Media_CG_H_02',
 'Media_SG_H',
 'Media_SG_H_02',
 'Media_Ptos_H',
 'Media_GM_H',
 'Media_SG_H_01',
 'CV_Ptos_A',
 'Media_GS_A',
 'DifAbs_DrawAway',
 'CV_CG_A_02',
 'CV_CG_A_01',
 'CV_GS_H',
 'CV_CG_A_00',
 'CV_SG_H',
 'CV_GM_A',
 'Media_CG_A_01',
 'CV_VG_A',
 'Media_VG_A',
 'Media_CG_A_00',
 'CV_SG_H_02',
 'CV_SG_H_01',
 'CV_SG_A_02',
 'CV_SG_A_01',
 'Media_CG_H_01',
 'Media_VG_H',
 'Media_CG_H_00',
 'CV_CG_H_00',
 'CV_SG_A',
 'CV_VG_H',
 'CV_GS_A',
 'CV_GM_H',
 'CV_CG_H_02',
 'CV_CG_H_01',
 'Media_GM_A',
 'Media_GS_H',
 'DifPer_HomeAway',
 'Media_SG_A_02',
 'Media_CG_A_02',
 'Media_Ptos_A',
 'Media_SG_A_01',
 'CV_Ptos_H',
 'Media_SG_A',
 'p_D',
 'DifPer_DrawAway',
 'H_A',
 'D_A',
 'H_D',
 'p_A'
 ]

        label = 'Back_Home'

        Back_Home = setup(data = treino[features], target = treino[label], session_id=123, verbose = False)

        df_models = models()
        modelos = df_models.index.tolist()

        lista = []
        for i in modelos:

            modelo = create_model(i, verbose=False)
            df0 = predict_model(modelo, teste, verbose=False)

            try:
                filtro = df0.prediction_label == 1
                df0 = df0[filtro]
                df0 = drop_reset_index(df0)

                df0['Profit_acu'] = df0.Profit.cumsum()
                n_apostas = df0.shape[0]
                profit = round(df0.Profit_acu.tail(1).item(), 2)
                ROI = round((df0.Profit_acu.tail(1) / n_apostas * 100).item(), 2)
                drawdown = df0['Profit_acu'] - df0['Profit_acu'].cummax()
                drawdown_maximo = round(drawdown.min(), 2)
                winrate_medio = round((df0['Profit'] > 0).mean() * 100, 2)
                desvio_padrao = round(df0['Profit'].std(), 2)

                metodos = {}

                metodos["League"] = liga
                metodos["Método"] = i
                metodos["Profit"] = profit
                metodos["Nº de Jogos"] = n_apostas
                metodos["ROI"] = ROI
                metodos["DrawDown"] = drawdown_maximo
                metodos["WinRate"] = winrate_medio
                metodos["DesvPad"] = desvio_padrao

                lista.append(metodos)
                print(f'Modelo: {i}, Profit: {profit}')

            except:
                print(f'Erro no Modelo {modelo}')

        df = pd.DataFrame(lista)
        df = df.sort_values(by='ROI',ascending=False)
        flt = (df['Nº de Jogos'] >= 100) & (df['ROI'] >= 5)
        df = df[flt]
        df = drop_reset_index(df)
        display(df.head())
        print('')
        df_Modelos = pd.concat([df_Modelos, df], ignore_index=True)

        try:
            best_method = df['Método'][1]
            Back_Home_PyCaret = create_model(best_method, verbose=False)

            jogos_do_dia = pd.read_csv('https://github.com/futpythontrader/YouTube/blob/main/Jogos_do_Dia_FlashScore/'+str(dia)+'_Jogos_do_Dia_FlashScore.csv?raw=true')
            jogos_do_dia = jogos_do_dia[['League','Date','Time','Home','Away','Odd_H','Odd_D','Odd_A']]
            Jogos_do_Dia = jogos_do_dia[jogos_do_dia['League'].isin(lista_ligas) == True]
            Jogos_do_Dia = drop_reset_index(Jogos_do_Dia)

            Jogos_do_Dia['p_H'] = 1 / Jogos_do_Dia['Odd_H']
            Jogos_do_Dia['p_D'] = 1 / Jogos_do_Dia['Odd_D']
            Jogos_do_Dia['p_A'] = 1 / Jogos_do_Dia['Odd_A']

            desvio_padrao = Jogos_do_Dia[['Odd_H','Odd_D','Odd_A']].std(axis=1)
            media = Jogos_do_Dia[['Odd_H','Odd_D','Odd_A']].mean(axis=1)
            CV_Odds = desvio_padrao / media
            Jogos_do_Dia['CV_Odds'] = CV_Odds

            Jogos_do_Dia['H_D'] = Jogos_do_Dia['Odd_H'] / Jogos_do_Dia['Odd_D']
            Jogos_do_Dia['H_A'] = Jogos_do_Dia['Odd_H'] / Jogos_do_Dia['Odd_A']
            Jogos_do_Dia['D_H'] = Jogos_do_Dia['Odd_D'] / Jogos_do_Dia['Odd_H']
            Jogos_do_Dia['D_A'] = Jogos_do_Dia['Odd_D'] / Jogos_do_Dia['Odd_A']
            Jogos_do_Dia['A_H'] = Jogos_do_Dia['Odd_A'] / Jogos_do_Dia['Odd_H']
            Jogos_do_Dia['A_D'] = Jogos_do_Dia['Odd_A'] / Jogos_do_Dia['Odd_D']

            Jogos_do_Dia['DifAbs_HomeAway'] = np.abs(Jogos_do_Dia['Odd_H'] - Jogos_do_Dia['Odd_A'])
            Jogos_do_Dia['DifAbs_HomeDraw'] = np.abs(Jogos_do_Dia['Odd_H'] - Jogos_do_Dia['Odd_D'])
            Jogos_do_Dia['DifAbs_DrawAway'] = np.abs(Jogos_do_Dia['Odd_D'] - Jogos_do_Dia['Odd_A'])

            Jogos_do_Dia['Angle_HomeAway'] = np.degrees(np.arctan((Jogos_do_Dia['Odd_A'] - Jogos_do_Dia['Odd_H']) / 2))
            Jogos_do_Dia['Angle_HomeDraw'] = np.degrees(np.arctan((Jogos_do_Dia['Odd_D'] - Jogos_do_Dia['Odd_H']) / 2))
            Jogos_do_Dia['Angle_DrawAway'] = np.degrees(np.arctan((Jogos_do_Dia['Odd_A'] - Jogos_do_Dia['Odd_D']) / 2))

            Jogos_do_Dia['DifPer_HomeAway'] = np.abs((Jogos_do_Dia['Odd_H'] - Jogos_do_Dia['Odd_A'])) / Jogos_do_Dia['Odd_A']
            Jogos_do_Dia['DifPer_HomeDraw'] = np.abs((Jogos_do_Dia['Odd_H'] - Jogos_do_Dia['Odd_D'])) / Jogos_do_Dia['Odd_D']
            Jogos_do_Dia['DifPer_DrawAway'] = np.abs((Jogos_do_Dia['Odd_D'] - Jogos_do_Dia['Odd_A'])) / Jogos_do_Dia['Odd_A']

            base = base[base['League'].isin(lista_ligas) == True]
            base = drop_reset_index(base)

            n_per = 5

            base['p_H'] = 1 / base['Odd_H']
            base['p_D'] = 1 / base['Odd_D']
            base['p_A'] = 1 / base['Odd_A']

            desvio_padrao = base[['Odd_H','Odd_D','Odd_A']].std(axis=1)
            media = base[['Odd_H','Odd_D','Odd_A']].mean(axis=1)
            CV_Odds = desvio_padrao / media
            base['CV_Odds'] = CV_Odds

            base['Ptos_H'] = np.where(base['Goals_H'] >  base['Goals_A'], 3,
                        np.where(base['Goals_H'] == base['Goals_A'], 1, 0))
            base['Ptos_A'] = np.where(base['Goals_H'] > base['Goals_A'], 0,
                        np.where(base['Goals_H'] == base['Goals_A'], 1, 3))

            base['Media_Ptos_H'] = base.groupby('Home')['Ptos_H'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
            base['Media_Ptos_A'] = base.groupby('Away')['Ptos_A'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
            base['DesvPad_Ptos_H'] = base.groupby('Home')['Ptos_H'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
            base['DesvPad_Ptos_A'] = base.groupby('Away')['Ptos_A'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
            base['CV_Ptos_H'] = base['DesvPad_Ptos_H'] / base['Media_Ptos_H']
            base['CV_Ptos_A'] = base['DesvPad_Ptos_A'] / base['Media_Ptos_A']

            base['Media_GM_H'] = base.groupby('Home')['Goals_H'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
            base['Media_GM_A'] = base.groupby('Away')['Goals_A'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
            base['DesvPad_GM_H'] = base.groupby('Home')['Goals_H'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
            base['DesvPad_GM_A'] = base.groupby('Away')['Goals_A'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
            base['CV_GM_H'] = base['DesvPad_GM_H'] / base['Media_GM_H']
            base['CV_GM_A'] = base['DesvPad_GM_A'] / base['Media_GM_A']

            base['Media_GS_H'] = base.groupby('Home')['Goals_A'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
            base['Media_GS_A'] = base.groupby('Away')['Goals_H'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
            base['DesvPad_GS_H'] = base.groupby('Home')['Goals_A'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
            base['DesvPad_GS_A'] = base.groupby('Away')['Goals_H'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
            base['CV_GS_H'] = base['DesvPad_GS_H'] / base['Media_GS_H']
            base['CV_GS_A'] = base['DesvPad_GS_A'] / base['Media_GS_A']

            base['SG_H'] = base['Goals_H'] - base['Goals_A']
            base['SG_A'] = base['Goals_A'] - base['Goals_H']
            base['Media_SG_H'] = base.groupby('Home')['SG_H'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
            base['Media_SG_A'] = base.groupby('Away')['SG_A'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
            base['DesvPad_SG_H'] = base.groupby('Home')['SG_H'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
            base['DesvPad_SG_A'] = base.groupby('Away')['SG_A'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
            base['CV_SG_H'] = base['DesvPad_SG_H'] / base['Media_SG_H']
            base['CV_SG_A'] = base['DesvPad_SG_A'] / base['Media_SG_A']

            base['SG_H_01'] = (base['Goals_H'] - base['Goals_A']) / base['p_H']
            base['SG_A_01'] = (base['Goals_A'] - base['Goals_H']) / base['p_A']
            base['Media_SG_H_01'] = base.groupby('Home')['SG_H_01'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
            base['Media_SG_A_01'] = base.groupby('Away')['SG_A_01'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
            base['DesvPad_SG_H_01'] = base.groupby('Home')['SG_H_01'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
            base['DesvPad_SG_A_01'] = base.groupby('Away')['SG_A_01'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
            base['CV_SG_H_01'] = base['DesvPad_SG_H_01'] / base['Media_SG_H_01']
            base['CV_SG_A_01'] = base['DesvPad_SG_A_01'] / base['Media_SG_A_01']

            base['SG_H_02'] = (base['Goals_H'] - base['Goals_A']) / base['p_A']
            base['SG_A_02'] = (base['Goals_A'] - base['Goals_H']) / base['p_H']
            base['Media_SG_H_02'] = base.groupby('Home')['SG_H_02'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
            base['Media_SG_A_02'] = base.groupby('Away')['SG_A_02'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
            base['DesvPad_SG_H_02'] = base.groupby('Home')['SG_H_02'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
            base['DesvPad_SG_A_02'] = base.groupby('Away')['SG_A_02'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
            base['CV_SG_H_02'] = base['DesvPad_SG_H_02'] / base['Media_SG_H_02']
            base['CV_SG_A_02'] = base['DesvPad_SG_A_02'] / base['Media_SG_A_02']

            base['VG_H'] = base['Goals_H'] * base['p_A']
            base['VG_A'] = base['Goals_A'] * base['p_H']
            base['Media_VG_H'] = base.groupby('Home')['VG_H'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
            base['Media_VG_A'] = base.groupby('Away')['VG_A'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
            base['DesvPad_VG_H'] = base.groupby('Home')['VG_H'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
            base['DesvPad_VG_A'] = base.groupby('Away')['VG_A'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
            base['CV_VG_H'] = base['DesvPad_VG_H'] / base['Media_VG_H']
            base['CV_VG_A'] = base['DesvPad_VG_A'] / base['Media_VG_A']

            base['CG_H_00'] = base['p_H'] / base['Goals_H']
            base['CG_A_00'] = base['p_A'] / base['Goals_A']
            base.replace(np.inf, 1, inplace=True)
            base['Media_CG_H_00'] = base.groupby('Home')['CG_H_00'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
            base['Media_CG_A_00'] = base.groupby('Away')['CG_A_00'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
            base['DesvPad_CG_H_00'] = base.groupby('Home')['CG_H_00'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
            base['DesvPad_CG_A_00'] = base.groupby('Away')['CG_A_00'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
            base['CV_CG_H_00'] = base['DesvPad_CG_H_00'] / base['Media_CG_H_00']
            base['CV_CG_A_00'] = base['DesvPad_CG_A_00'] / base['Media_CG_A_00']

            base['CG_H_01'] = base['Goals_H'] / base['p_H']
            base['CG_A_01'] = base['Goals_A'] / base['p_A']
            base['Media_CG_H_01'] = base.groupby('Home')['CG_H_01'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
            base['Media_CG_A_01'] = base.groupby('Away')['CG_A_01'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
            base['DesvPad_CG_H_01'] = base.groupby('Home')['CG_H_01'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
            base['DesvPad_CG_A_01'] = base.groupby('Away')['CG_A_01'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
            base['CV_CG_H_01'] = base['DesvPad_CG_H_01'] / base['Media_CG_H_01']
            base['CV_CG_A_01'] = base['DesvPad_CG_A_01'] / base['Media_CG_A_01']

            base['CG_H_02'] = (base['Goals_H'] / 2) + (base['p_H'] / 2)
            base['CG_A_02'] = (base['Goals_A'] / 2) + (base['p_A'] / 2)
            base['Media_CG_H_02'] = base.groupby('Home')['CG_H_02'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
            base['Media_CG_A_02'] = base.groupby('Away')['CG_A_02'].rolling(window=n_per, min_periods=n_per).mean().reset_index(0,drop=True)
            base['DesvPad_CG_H_02'] = base.groupby('Home')['CG_H_02'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
            base['DesvPad_CG_A_02'] = base.groupby('Away')['CG_A_02'].rolling(window=n_per, min_periods=n_per).std().reset_index(0,drop=True)
            base['CV_CG_H_02'] = base['DesvPad_CG_H_02'] / base['Media_CG_H_02']
            base['CV_CG_A_02'] = base['DesvPad_CG_A_02'] / base['Media_CG_A_02']

            base['H_D'] = base['Odd_H'] / base['Odd_D']
            base['H_A'] = base['Odd_H'] / base['Odd_A']
            base['D_H'] = base['Odd_D'] / base['Odd_H']
            base['D_A'] = base['Odd_D'] / base['Odd_A']
            base['A_H'] = base['Odd_A'] / base['Odd_H']
            base['A_D'] = base['Odd_A'] / base['Odd_D']

            base['DifAbs_HomeAway'] = np.abs(base['Odd_H'] - base['Odd_A'])
            base['DifAbs_HomeDraw'] = np.abs(base['Odd_H'] - base['Odd_D'])
            base['DifAbs_DrawAway'] = np.abs(base['Odd_D'] - base['Odd_A'])

            base['Angle_HomeAway'] = np.degrees(np.arctan((base['Odd_A'] - base['Odd_H']) / 2))
            base['Angle_HomeDraw'] = np.degrees(np.arctan((base['Odd_D'] - base['Odd_H']) / 2))
            base['Angle_DrawAway'] = np.degrees(np.arctan((base['Odd_A'] - base['Odd_D']) / 2))

            base['DifPer_HomeAway'] = np.abs((base['Odd_H'] - base['Odd_A'])) / base['Odd_A']
            base['DifPer_HomeDraw'] = np.abs((base['Odd_H'] - base['Odd_D'])) / base['Odd_D']
            base['DifPer_DrawAway'] = np.abs((base['Odd_D'] - base['Odd_A'])) / base['Odd_A']

            colunas_deletar = ['Ptos_H','Ptos_A','DesvPad_Ptos_H','DesvPad_Ptos_A',
                            'DesvPad_GM_H','DesvPad_GM_A',
                            'DesvPad_GS_H','DesvPad_GS_A',
                            'SG_H','SG_A','DesvPad_SG_H','DesvPad_SG_A',
                            'SG_H_01','SG_A_01','DesvPad_SG_H_01','DesvPad_SG_A_01',
                            'SG_H_02','SG_A_02','DesvPad_SG_H_02','DesvPad_SG_A_02',
                            'VG_H','VG_A','DesvPad_VG_H','DesvPad_VG_A',
                            'CG_H_00','CG_A_00','DesvPad_CG_H_00','DesvPad_CG_A_00',
                            'CG_H_01','CG_A_01','DesvPad_CG_H_01','DesvPad_CG_A_01',
                            'CG_H_02','CG_A_02','DesvPad_CG_H_02','DesvPad_CG_A_02']

            base = base.drop(colunas_deletar, axis=1)
            base = drop_reset_index(base)

            base_H = base[['Home',
                            'Media_Ptos_H','CV_Ptos_H',
                            'Media_GM_H','CV_GM_H',
                            'Media_GS_H','CV_GS_H',
                            'Media_SG_H','CV_SG_H',
                            'Media_SG_H_01','CV_SG_H_01',
                            'Media_SG_H_02','CV_SG_H_02',
                            'Media_VG_H','CV_VG_H',
                            'Media_CG_H_00','CV_CG_H_00',
                            'Media_CG_H_01','CV_CG_H_01',
                            'Media_CG_H_02','CV_CG_H_02']]

            base_A = base[['Away',
                            'Media_Ptos_A','CV_Ptos_A',
                            'Media_GM_A','CV_GM_A',
                            'Media_GS_A','CV_GS_A',
                            'Media_SG_A','CV_SG_A',
                            'Media_SG_A_01','CV_SG_A_01',
                            'Media_SG_A_02','CV_SG_A_02',
                            'Media_VG_A','CV_VG_A',
                            'Media_CG_A_00','CV_CG_A_00',
                            'Media_CG_A_01','CV_CG_A_01',
                            'Media_CG_A_02','CV_CG_A_02']]

            ultima_base_H = base_H.groupby('Home').last().reset_index()
            ultima_base_A = base_A.groupby('Away').last().reset_index()

            df = pd.merge(Jogos_do_Dia, ultima_base_H, how='left', left_on='Home', right_on='Home')
            df = pd.merge(df, ultima_base_A, how='left', left_on='Away', right_on='Away')

            df = drop_reset_index(df)

            df1 = predict_model(Back_Home_PyCaret, data=df)
            filtro = df1.prediction_label == 1
            Entradas = df1[filtro]
            Entradas = drop_reset_index(Entradas)
            Entradas = Entradas[['League','Date','Time','Home','Away','Odd_H','Odd_D','Odd_A']]
            df_Entradas = pd.concat([df_Entradas, Entradas], ignore_index=True)
            print('');display(df_Entradas)
        except:
            pass

In [ ]:
try:
    ligas_entradas = df_Modelos['League'].unique()
    display(ligas_entradas)
except:
    pass

In [ ]:
try:
    ligas_entradas = df_Modelos['League'].unique()
    df_Entradas = df_Entradas[df_Entradas['League'].isin(ligas_entradas) == True]
    df_Entradas = df_Entradas.sort_values(by='Time')
    df_Entradas = drop_reset_index(df_Entradas)
    display(df_Entradas)
except:
    pass

### Entradas

In [ ]:
flt = base_orig.Date == str(dia)
base_today = base_orig[flt]
base_today = base_today[['League','Home','Away','Goals_H','Goals_A','Goals_Minutes_Home','Goals_Minutes_Away']]
base_today = drop_reset_index(base_today)

Entradas = pd.merge(df_Entradas, base_today, on=['League','Home', 'Away'])
Entradas = drop_reset_index(Entradas)
Entradas['Profit'] = np.where((Entradas['Goals_H'] >  Entradas['Goals_A']), Entradas['Odd_H']-1, -1)
Entradas['Profit_Acu'] = Entradas['Profit'].cumsum()
Entradas